# CUDA/T4 V-14 core: qualification, validation, and timing

This notebook is self-contained for a fresh Google Colab runtime. It performs only:

- fresh CUDA/T4 unbalanced and slack transcript qualifications;
- full-domain, non-value core validation;
- non-value end-to-end core throughput measurement; and
- generation of a **plan**.

It deliberately has no `--run-batch` cell. Select a **T4 GPU** under **Runtime → Change runtime type** before running the hardware check.

In [ ]:
import shutil
import subprocess

assert shutil.which('nvidia-smi'), 'No GPU backend: select a T4 GPU runtime, reconnect, then rerun.'
assert shutil.which('nvcc'), 'nvcc is unavailable: this is not the required CUDA/T4 runtime.'

print(subprocess.check_output(['nvidia-smi', '--query-gpu=name,driver_version', '--format=csv,noheader'], text=True))
print(subprocess.check_output(['nvcc', '--version'], text=True))

In [ ]:
from google.colab import files
from pathlib import Path
import hashlib
import tarfile

EXPECTED_BUNDLE_SHA256 = 'bde46c5965a51f7ece34b13da1f7ed3b4859db36518a5090a680a9c8e5b37d93'
uploaded = files.upload()  # Select g6_cuda_t4_v14_core_bundle_v2.tar.gz
archive = Path(next(iter(uploaded)))
actual = hashlib.sha256(archive.read_bytes()).hexdigest()
print(f'{archive.name}: {actual}')
assert actual == EXPECTED_BUNDLE_SHA256, 'Wrong or incomplete bundle; do not continue.'

with tarfile.open(archive, 'r:gz') as bundle:
    bundle.extractall('/content', filter='data')

ROOT = Path('/content/project')
assert (ROOT / 'research/g6/run_cuda_v14_core_batches.py').is_file()
print('Core bundle extracted to', ROOT)

In [ ]:
from google.colab import drive

drive.mount('/content/gdrive')

# Change only RUN_ID if this notebook is deliberately rerun in another same-day session.
RUN_ID = 'run3'
OUT = Path(f'/content/gdrive/MyDrive/g6_cuda_t4_v14_core_20260921_{RUN_ID}')
assert not OUT.exists(), f'{OUT} already exists; change RUN_ID instead of mixing sessions.'
OUT.mkdir(parents=True)
print('Persistent output directory:', OUT)

In [ ]:
expected_sources = {
    'research/g6/cuda_qualification/cuda_v14_core_batch_engine.cu': 'ac9cb5cb59320b9d9d1131f60fd4b954ec05ddc843dd94b6cf8ceab785d9e0c5',
    'research/g6/cuda_qualification/cuda_v14_core_batch.py': '36028657cd6483192b4734471ca7d0cc27c0d4de7a17b02bd7f054845ca00c3c',
    'research/g6/cuda_qualification/v14_core_batch_contract.py': '6601c61f89efdddd3a404e1571facc842baccc753efc4414ef8aa4dcf34c175b',
}
for relative, wanted in expected_sources.items():
    actual = hashlib.sha256((ROOT / relative).read_bytes()).hexdigest()
    assert actual == wanted, (relative, actual, wanted)
print('Repaired core source hashes: PASS')

In [ ]:
import json
import os

ENV = dict(os.environ, PYTHONDONTWRITEBYTECODE='1')

def run_json(name, arguments):
    destination = OUT / name
    print('Running:', ' '.join(arguments))
    with destination.open('w', encoding='utf-8') as stream:
        result = subprocess.run(arguments, cwd=ROOT, env=ENV, stdout=stream, stderr=subprocess.PIPE, text=True)
    if result.returncode:
        raise RuntimeError('Command failed: ' + ' '.join(arguments) + '\n\n' + result.stderr[-6000:])
    return json.loads(destination.read_text(encoding='utf-8'))

# New qualification records are required for this particular T4 session.
unbalanced = run_json('unbalanced_qualification.json', [
    'python3', 'research/g6/run_cuda_energy_crosscheck.py', '--run'
])
assert unbalanced['verdict'] == 'pass', unbalanced

slack = run_json('slack_qualification.json', [
    'python3', 'research/g6/run_cuda_slack_energy_crosscheck.py', '--run'
])
assert slack['verdict'] == 'pass', slack

validation = run_json('v14_core_validation.json', [
    'python3', 'research/g6/run_cuda_v14_core_batches.py', '--validate',
    '--unbalanced-record', str(OUT / 'unbalanced_qualification.json'),
    '--slack-record', str(OUT / 'slack_qualification.json'),
])
assert validation['verdict'] == 'pass', validation
assert all(value == 0 for value in validation['validation']['totals'].values()), validation
print('Core validation: PASS')

In [ ]:
# This measures the exact result-production path but discards all constructed records.
# No V-14 scientific output is retained in this cell.
measurement_run = run_json('v14_core_measurement_2x_0_4096.json', [
    'python3', 'research/g6/run_cuda_v14_core_batches.py', '--measure',
    '--scale', '2x', '--start', '0', '--stop', '4096',
    '--unbalanced-record', str(OUT / 'unbalanced_qualification.json'),
    '--slack-record', str(OUT / 'slack_qualification.json'),
])
assert measurement_run['verdict'] == 'measured', measurement_run
measurement = measurement_run['throughput_measurement']
(OUT / 'v14_core_throughput_measurement.json').write_text(
    json.dumps(measurement, sort_keys=True, separators=(',', ':')), encoding='utf-8'
)
sample = measurement['samples'][0]
print('Core measurement: PASS')
print('coordinates:', sample['coordinates'])
print('wall seconds:', sample['wall_seconds'])
print('coordinate seconds:', sample['coordinate_seconds'])
print('setup seconds:', sample['setup_seconds'])

In [ ]:
plan = run_json('v14_2x_core_batch_plan.json', [
    'python3', 'research/g6/plan_v14_cuda_core_batches.py', 'plan', '--scale', '2x',
    '--validation-record', str(OUT / 'v14_core_validation.json'),
    '--measurement', str(OUT / 'v14_core_throughput_measurement.json'),
])
sizing = plan['sizing']
print('Plan acceptance:', sizing['measurement_acceptance_basis'])
print('Coordinates per batch:', sizing['coordinates_per_batch'])
print('Batch count:', sizing['batch_count'])
print('Plan hash:', plan['content_sha256'])
print('\nStop here. Send the three core JSON files and this summary for review; do not run --run-batch.')

## Stop point

After the plan cell, retain these Drive files for review:

- `v14_core_validation.json`
- `v14_core_measurement_2x_0_4096.json`
- `v14_2x_core_batch_plan.json`

This notebook intentionally does not contain a core batch producer, merger, aggregation, rate calculation, or conclusion.